## General requirements & preparation

This timeseries script uses functions from atlite to generate weather timeseries for GENeSYS-MOD. It uses the ERA5 weather dataset, which is downloaded and transformed locally.
Please be aware that for the entire European dataset, a large amount of system memory is required (at least 24GB of RAM).


To use this script, you need to have the following packages installed:
- numpy
- matplotlib
- seaborn
- pandas / geopandas
- scikit-learn
- cartopy
- xarray
- atlite

In addition, in order to be able to load the ERA5 cutouts, you need an API key from https://cds.climate.copernicus.eu/user/register?destination=%2F%23!%2Fhome

To activate the API key, follow the instructions at https://cds.climate.copernicus.eu/api-how-to

For more information on how to run atlite, please refer to https://atlite.readthedocs.io/en/latest/

In [ ]:
from functions import *

In [ ]:
onshore_turbine = 'Vestas_V112_3MW'
offshore_turbine = 'Vestas_V164_7MW_offshore'
solar_panel = 'CSi'

# Region polygons to analyse: a geojson with a 'region' column. Results are produced
# per region in this file. This small example ships a single 'Germany' region
# (geodata/example_germany.geojson) so the notebook runs without large downloads.
geo_file = "geodata/example_germany.geojson"

# Offshore needs an EEZ polygon + a bathymetry grid for the region. The example
# ships small Germany files (German EEZ from Marine Regions v12; bathymetry from
# NOAA SRTM15+). Set both to None to disable the offshore steps.
offshore_file = "geodata/example_germany_eez.geojson"
bathymetry_file = "geodata/example_germany_bathymetry.nc"

timeframe = "2018-01-01"   # one day keeps this example small. format: 2018 OR 2018-01 OR 2018-01-01
filename = "germany"       # name for the downloaded cutout; change per run/year/region
output_dir = create_output_folder(timeframe)   # creates an output folder if it does not exist yet

# admin is only used by the country/NUTS workflow; it is ignored in region-geojson mode
admin = 0

# Subset of region names from geo_file to keep. Leave the list empty to use all regions.
regions = []

# cutout coordinates. Leave empty if the cutout should be taken from the geo_file regions.
cutout_north_west = []
cutout_south_east = []

In [ ]:
# PV slope/azimuth for utility-scale (ground-mounted) PV.
pv_slope = 36.7
pv_azimuth = 180

# Optimal tilt: if True, utility-scale PV uses a per-coordinate optimal fixed
# tilt (function of latitude) and equator-facing azimuth instead of the single
# pv_slope/pv_azimuth above. Cheap closed-form rule (no per-angle yield search).
pv_optimal_tilt = True

# Rooftop PV is installed differently (roof pitch / mixed orientations), so it
# keeps its own fixed slope/azimuth for the separate pv_rooftop timeseries.
rooftop_pv_slope = 25
rooftop_pv_azimuth = 180

# Usable-site thresholds: minimum developable share of a cutout cell for it
# to count as a usable location for the capacity-factor timeseries. Land is
# lenient (1%); rooftop is the built-up fraction (small everywhere), so it
# needs a higher bar to drop near-empty cells - raise if still too many.
usable_threshold = 0.01
usable_threshold_rooftop = 0.05

In [ ]:
# Horizontal and vertical spacing (in degrees) between coordinates. 0.25 is a
# reasonable default; larger values give a coarser, faster cutout.
dx_step = 0.25
dy_step = 0.25

## Region Defitinition & Cutout preparation

In this block, you need to define your coordinates, either by using your own fixed coordinates (by defining custom bounds), or using the automatic .geojson mapping for the country of your choice.

Be aware that the first preparation of the cutout will take significant time!

In [ ]:
# Pre-check the expected cutout size BEFORE downloading anything: grid cells,
# hourly time steps and total data points. Uses the same regions/geo_file/resolution
# as get_cutout below, so adjust dx/dy or the region selection if it is too large.
estimate_cutout_size(timeframe, regions=regions, geo_file=geo_file, dx=dx_step, dy=dy_step)

In [ ]:
cutout = get_cutout(filename, 
                    timeframe, 
                    regions=regions, 
                    geo_file=geo_file, 
                    dx=dx_step, 
                    dy=dy_step)

Plotting your cutout to ensure that everything is correct!

You can add zoom=True/False to zoom into your selection and use size to set the size of the graph if needed.

In [ ]:
plot_country_map(cutout) # plot_country_map(cutout,zoom=False,size=8)

## General settings and functions

Always run these to create the relevant data entries.

In [ ]:
coords_onshore, coords_offshore = get_coords(cutout, 
                                             regions, 
                                             geo_file, 
                                             admin=admin,
                                             offshore_file=offshore_file,
                                             bathymetry_file=bathymetry_file)

## Functions for timeseries generation

In [ ]:
pv_inf, pv_avg, pv_opt = pv_capacity_factors(cutout, 
                                             coords_onshore, 
                                             solar_panel,
                                             pv_slope=pv_slope,
                                             pv_azimuth=pv_azimuth, 
                                             optimal_tilt=pv_optimal_tilt,
                                             timeframe=timeframe, 
                                             filename=filename,
                                             write_raw_data=False,
                                             output_dir=output_dir)

In [ ]:
pv_tra = pv_capacity_factors(cutout, 
                             coords_onshore, 
                             solar_panel,
                             tracking= "horizontal",
                             pv_azimuth=0, #refers to oriantation of the axis in this case
                             timeframe=timeframe, 
                             filename=filename,
                             write_raw_data=False,
                             output_dir=output_dir)

In [ ]:
display(pv_inf.mean())
display(pv_avg.mean())
display(pv_opt.mean())

In [ ]:
display(pv_tra.mean())

In [ ]:
wind_onshore_inf, wind_onshore_avg, wind_onshore_opt = wind_onshore_capacity_factors(cutout, 
                                                                                     coords_onshore, 
                                                                                     onshore_turbine,  
                                                                                     timeframe=timeframe, 
                                                                                     filename=filename,
                                                                                     write_raw_data=False, # determines if you want to have the aggregated outputs per administrative region or the full dataset of each individual coordinate (default: False)
                                                                                     output_dir=output_dir)

In [ ]:
display(wind_onshore_inf.mean()) 
display(wind_onshore_avg.mean()) 
display(wind_onshore_opt.mean())

In [ ]:
# Offshore wind capacity factors. Requires offshore_file + bathymetry_file set
# (coords_offshore non-empty); skipped in this example where offshore is disabled.
if coords_offshore is not None and len(coords_offshore) > 0:
    wind_offshore_shallow, wind_offshore_transitional, wind_offshore_deep = wind_offshore_capacity_factors(
        cutout, coords_offshore, offshore_turbine,
        timeframe=timeframe, filename=filename,
        write_raw_data=False, output_dir=output_dir)
else:
    wind_offshore_shallow = wind_offshore_transitional = wind_offshore_deep = None
    print("Offshore disabled (offshore_file / bathymetry_file are None) - skipping.")

In [ ]:
if wind_offshore_shallow is not None:
    display(wind_offshore_shallow.mean())
    display(wind_offshore_transitional.mean())
    display(wind_offshore_deep.mean())

In [ ]:
df_heatpump_ground_cop, df_cooling, df_heating, df_heatpump_cop = temperature_timeseries(cutout,
                                                                                        coords_onshore,
                                                                                        timeframe=timeframe,
                                                                                        filename=filename,
                                                                                        write_raw_data=False, # determines if you want to have the aggregated outputs per administrative region or the full dataset of each individual coordinate (default: False)
                                                                                        output_dir=output_dir)

In [ ]:
display(df_heatpump_ground_cop.mean())
display(df_heatpump_cop.mean())
display(df_cooling.mean())
display(df_heating.mean())

# GIS-based renewable energy potentials

##### This part is still somewhat experimental. It uses functionality from the atlite package to calculate available capacities for utility-scale PV, onshore wind, and rooftop PV installations.
##### It needs a land-cover raster and a protected-areas file for your region (set in the cell below). These layers are large and are not shipped with this example.

In [ ]:
# Region-agnostic: use the SAME region polygons as the timeseries (onshore),
# not the country/NUTS workflow. Returns region shapes + names.
shapes, regions_name_en = get_region_shapes(geo_file, regions)
#shapes, regions_name_en = get_region_shapes(geo_file, ['California'])

In [ ]:
# --- Exclusion layers ------------------------------------------------------
# The builder functions are region-agnostic; only the paths/codes below are
# region-specific. Land cover here is CORINE (Copernicus, 100 m, EPSG:3035),
# shipped small for this Germany example. Swap for your region's land cover.
#   https://land.copernicus.eu/en/products/corine-land-cover
LANDCOVER = "geodata/corine.tif"
LANDCOVER_CRS = None                  # CORINE embeds EPSG:3035

# CORINE CLC raster legend (grid value -> class). Land availability for
# utility-scale PV / onshore wind. Exclude:
#   1-11           artificial surfaces (urban, industry, transport, mines, etc.)
#   12-17,19,20    arable + permanent crops (protect prime cropland)
#   23-25          forest (broad-leaved, coniferous, mixed)
#   34             glaciers / perpetual snow
#   35-44          wetlands + water (marshes, bogs, water bodies, sea)
# Keep: 18 pastures, 21 agriculture-with-natural-veg, 22 agro-forestry, and
# 26-33 natural (grassland, moors/heath, sclerophyllous, transitional woodland-
# shrub, beaches/dunes, bare rock, sparse veg, burnt). Pasture/grassland +
# agrivoltaics on mixed ag are usable; dense crops + forest are not.
landcover_exclude = (list(range(1, 12))          # artificial
                     + [12, 13, 14, 15, 16, 17, 19, 20]   # arable + permanent crops
                     + [23, 24, 25]              # forest
                     + [34]                      # glaciers
                     + list(range(35, 45)))      # wetlands + water

# Protected areas: e.g. WDPA (protectedplanet.net) or a national dataset. Optional
# for this example (None = land cover only). Set a path to enable.
PROTECTED       = None
PROTECTED_LAYER = None                # set if a multi-layer source (gpkg / gdb)
PROTECTED_QUERY = None                # e.g. "IUCN_CAT in ['Ia','Ib','II']"

excluder = make_land_excluder(
    LANDCOVER, landcover_exclude, raster_crs=LANDCOVER_CRS,
    protected_files=PROTECTED, protected_layer=PROTECTED_LAYER,
    protected_query=PROTECTED_QUERY,
)

# Rooftop proxy: keep ONLY the built-up CORINE classes (1 continuous + 2
# discontinuous urban fabric). Adjust to your land-cover legend.
cities = make_rooftop_excluder(LANDCOVER, [1, 2], raster_crs=LANDCOVER_CRS)

# --- Offshore exclusion (optional) -----------------------------------------
# A no-build mask for offshore (marine protected areas, shipping lanes, enclosed
# bays, ...). None = depth filter only. Provide a vector file to enable.
OFFSHORE_EXCLUDE_FILES = None
OFFSHORE_EXCLUDE_LAYER = None
OFFSHORE_EXCLUDE_QUERY = None

In [ ]:
pv_cap_per_sqkm = 100                   # MW
pv_percent_land_available = 0.03        # % of suitable area
wind_cap_per_sqkm = 27                  # MW
wind_percent_land_available = 0.03      # % of suitable area
rooftop_cap_per_sqkm = 100              # MW
rooftop_percent_area_available = 0.05   # installable share of built-up area

## Compute area availability (single pass, memory intensive)

This will calculate available surface areas for onshore, offshore, and rooftops using the excluders specified. Memory-intensive, since it calculates it all at once. See below for per-region loop version.

In [ ]:
AvailabilityMatrix = calculate_and_plot_available_area(admin,cutout,shapes,regions_name_en,excluder)

In [ ]:
AvailabilityMatrix_Rooftop = calculate_and_plot_available_rooftops(admin,cutout,shapes,regions_name_en,cities)

In [ ]:
output_df = calculate_capacity_potentials(cutout,coords_onshore,AvailabilityMatrix,AvailabilityMatrix_Rooftop,pv_cap_per_sqkm,pv_percent_land_available,wind_cap_per_sqkm,wind_percent_land_available,rooftop_cap_per_sqkm,rooftop_percent_area_available)

In [ ]:
# here is the resulting DataFrame containing all the potentials and areas that have been calculated
output_df

## Compute area availability (Memory-safe all-region run)

The cells above process every region at once, which rasterizes the land-cover map over the full extent and can exhaust RAM at 30 m. 
`calculate_potentials_per_region` processes one region at a time (bounded raster window), writes a per-region CSV plus a combined CSV, 
and returns one stitched availability map across all regions. Reuses the same `excluder`/`cities` defined above.

In [ ]:
# Memory-safe: process every region one at a time. Saves per-region + combined
# CSVs, returns one stitched availability map AND the usable coords (all regions)
# for the usable-site timeseries below: land (PV/onshore wind) and rooftop.
# Pass coords_offshore + offshore_exclude_* as well if offshore is enabled.
combined_potentials, stitched_land, stitched_rooftop, coords_onshore_usable, coords_rooftop_usable, coords_offshore_usable = calculate_potentials_per_region(
    cutout, geo_file, coords_onshore,
    excluder=excluder, cities=cities,
    regions=regions,
    coords_offshore=coords_offshore,
    usable_threshold=usable_threshold, usable_threshold_rooftop=usable_threshold_rooftop,
    offshore_exclude_files=OFFSHORE_EXCLUDE_FILES,
    offshore_exclude_layer=OFFSHORE_EXCLUDE_LAYER,
    offshore_exclude_query=OFFSHORE_EXCLUDE_QUERY,
    pv_cap_per_sqkm=pv_cap_per_sqkm, pv_percent_land_available=pv_percent_land_available,
    wind_cap_per_sqkm=wind_cap_per_sqkm, wind_percent_land_available=wind_percent_land_available,
    rooftop_cap_per_sqkm=rooftop_cap_per_sqkm, rooftop_percent_area_available=rooftop_percent_area_available,
    output_dir=output_dir, filename=filename,
)
combined_potentials

### Offshore wind potential

Reuses the EEZ-clipped, depth-classified offshore cells from `get_coords` (so only actual offshore zones are counted). Capacity density and available share can be set per depth class (shallow/transitional = fixed-bottom, deep = floating).

This section requires `offshore_file` and `bathymetry_file` to be set in the configuration cell. With both `None` (the example default) there are no offshore cells and these cells can be skipped.

In [ ]:
# Offshore wind potential per region and depth class. Requires offshore_file +
# bathymetry_file to be set (coords_offshore non-empty); skipped otherwise.
if coords_offshore is not None and len(coords_offshore) > 0:
    offshore_cap_per_sqkm = 5            # MW/km2
    offshore_percent_available = {"shallow": 0.10, "transitional": 0.10, "deep": 0.05}
    offshore_df = calculate_offshore_potentials(
        cutout, coords_offshore,
        offshore_cap_per_sqkm=offshore_cap_per_sqkm,
        offshore_percent_available=offshore_percent_available,
    )
    display(offshore_df)
else:
    print("Offshore disabled (offshore_file / bathymetry_file are None) - skipping.")

### Capacity-factor timeseries on usable sites only

Restrict the RES timeseries to cells that survive the exclusion layers. 
`usable_onshore_coords` keeps cutout cells whose availability > threshold (from the 
availability matrix); feed those into the normal PV / onshore-wind functions, which 
still split into opt/avg/inf. `usable_offshore_coords` keeps buildable offshore cells 
(depth within limits) for the offshore-wind timeseries.

In [ ]:
# Usable onshore sites across ALL regions come from the memory-safe loop above
# (coords_onshore_usable). For a quick single-region test instead, use:
#   coords_onshore_usable = usable_onshore_coords(AvailabilityMatrix, coords_onshore)
print(f"usable onshore cells: {len(coords_onshore_usable)} of {len(coords_onshore)}")
print(f"usable rooftop cells: {len(coords_rooftop_usable)} of {len(coords_onshore)}")

# Utility-scale PV on usable land (per-cell optimal tilt if pv_optimal_tilt):
pv_inf_u, pv_avg_u, pv_opt_u = pv_capacity_factors(cutout, coords_onshore_usable, solar_panel,
                                                   pv_slope=pv_slope, pv_azimuth=pv_azimuth,
                                                   optimal_tilt=pv_optimal_tilt,
                                                   timeframe=timeframe, filename=filename+"_usable",
                                                   write_raw_data=False, output_dir=output_dir)

# Rooftop PV: separate category, only rooftop-available cells, own fixed slope/azimuth
# (roofs dictate orientation, so no optimal-tilt here). Label "pv_rooftop".
pv_roof_inf, pv_roof_avg, pv_roof_opt = pv_capacity_factors(cutout, coords_rooftop_usable, solar_panel,
                                                            pv_slope=rooftop_pv_slope, pv_azimuth=rooftop_pv_azimuth,
                                                            tech_label="pv_rooftop",
                                                            timeframe=timeframe, filename=filename+"_usable",
                                                            write_raw_data=False, output_dir=output_dir)

# Onshore wind capacity factors on usable land:
wind_inf_u, wind_avg_u, wind_opt_u = wind_onshore_capacity_factors(cutout, coords_onshore_usable, onshore_turbine,
                                                                   timeframe=timeframe, filename=filename+"_usable",
                                                                   write_raw_data=False, output_dir=output_dir)

In [ ]:
# Usable offshore sites: from the loop above (coords_offshore_usable), already
# filtered by depth AND any offshore exclusion. Requires offshore to be enabled.
if coords_offshore is not None and len(coords_offshore) > 0:
    if coords_offshore_usable is None:
        coords_offshore_usable = usable_offshore_coords(coords_offshore,
            exclude_files=OFFSHORE_EXCLUDE_FILES, exclude_layer=OFFSHORE_EXCLUDE_LAYER,
            exclude_query=OFFSHORE_EXCLUDE_QUERY)
    print(f"usable offshore cells: {len(coords_offshore_usable)} of {len(coords_offshore)}")

    # Offshore wind capacity factors on usable sites only (shallow/transitional/deep):
    wind_off_shallow_u, wind_off_trans_u, wind_off_deep_u = wind_offshore_capacity_factors(
        cutout, coords_offshore_usable, offshore_turbine,
        timeframe=timeframe, filename=filename+"_usable",
        write_raw_data=False, output_dir=output_dir)

    # Map of the usable offshore cells, coloured by depth class:
    plot_offshore_map(coords_offshore_usable, shapes=shapes, offshore_file=offshore_file, color_by="depth_class")
else:
    print("Offshore disabled (offshore_file / bathymetry_file are None) - skipping.")